In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import global_mean_pool
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


In [ ]:
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from numpy import savetxt, loadtxt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input, Concatenate, Reshape, Conv1D, Flatten, Dense, Dropout, MultiHeadAttention, LayerNormalization, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from transformers import AutoTokenizer, AutoModel

from tqdm import tqdm
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dropout, Dense, concatenate, BatchNormalization



In [ ]:
def load_dataset(label_count):

    data = pd.read_csv("DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0')
    print(len(data))

    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Giữ đúng 39645 rows
    #data = data.iloc[:39645]
    total_rows = len(data)
    split_point = int(0.8 * total_rows)

    print(split_point)
    print(total_rows - split_point)

    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    

    if True:
        X_train_source = np.load("DATASET/X_train_source_minmaxsummean_10chunks_512.npy")
        X_test_source = np.load("DATASET/X_test_source_minmaxsummean_10chunks_512.npy")
        X_train_source = X_train_source.reshape(-1, 10, 4, 768)
        X_test_source = X_test_source.reshape(-1, 10, 4, 768)
        X_train_source = X_train_source[:, :, 0, :]
        X_test_source = X_test_source[:, :, 0, :]
        X_train_source = X_train_source.reshape(-1, 10 * 768)
        X_test_source = X_test_source.reshape(-1, 10 * 768)

    else:

        # =========================
        # CODEBERT EMBEDDING
        # =========================

        source_train = df_train['sourcecode'].tolist()
        source_test = df_test['sourcecode'].tolist()
        

        tokenizer = AutoTokenizer.from_pretrained("./codebert")
        model = AutoModel.from_pretrained("./codebert")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)


        model.eval()

        print("Generating CodeBERT embeddings...")

        X_train_source = get_codebert_embeddings(source_train, tokenizer, model, device)
        X_test_source = get_codebert_embeddings(source_test, tokenizer, model, device)
        
            
        # texts = df_train['sourcecode'].astype(str).tolist()

        # lengths = [len(tokenizer.tokenize(x)) for x in texts]

        # print("Average tokens:", np.mean(lengths))
        # print("Max tokens:", np.max(lengths))


        print("Load Features CodeBERT success!!")
        np.save("DATASET/X_train_source.npy", X_train_source)
        np.save("DATASET/X_test_source.npy", X_test_source)

    # =========================
    # OPCODE FEATURES (giữ nguyên)
    # =========================

    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # =========================
    # LABELS
    # =========================

    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")

    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels

In [ ]:
X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels = load_dataset(8)

In [ ]:
import torch
import pickle
import numpy as np
import torch.nn.functional as F

from torch.nn import Linear, BatchNorm1d
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================
# LOAD DATA
# =========================
input_file = 'DATASET/gnn_input.pkl'

with open(input_file, 'rb') as f:
    raw_data = pickle.load(f)

feature_list = raw_data['features']
adj_matrices = raw_data['adj_matrices']
label_list = raw_data['labels']

target_dim = 57
num_classes = len(label_list[0])

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim), dtype=np.float32)
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features.astype(np.float32)

def create_pyg_data(features, adj_matrix, label, target_dim):
    adj = torch.tensor(np.array(adj_matrix), dtype=torch.long)
    edge_index = adj.nonzero(as_tuple=False).t().contiguous()

    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(label, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)

data_list = [
    create_pyg_data(feat, adj, label, target_dim)
    for feat, adj, label in zip(feature_list, adj_matrices, label_list)
]

# nếu muốn shuffle trước khi split thì mở 2 dòng dưới
# indices = np.random.permutation(len(data_list))
# data_list = [data_list[i] for i in indices]

split_idx = int(len(data_list) * 0.8)
cfg_train_loader = DataLoader(data_list[:split_idx], batch_size=64, shuffle=False)
cfg_test_loader = DataLoader(data_list[split_idx:], batch_size=64, shuffle=False)


In [ ]:

class OpcodeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)     # input cho Embedding
        self.y = torch.tensor(y, dtype=torch.float32)  # cho BCELoss

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
class SourceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
opcode_train_dataset = OpcodeDataset(X_train_opcode, y_train)
opcode_test_dataset = OpcodeDataset(X_test_opcode, y_test)

opcode_train_loader = DataLoader(opcode_train_dataset, batch_size=64, shuffle=False)
opcode_test_loader  = DataLoader(opcode_test_dataset, batch_size=64, shuffle=False)

source_train_dataset = SourceDataset(X_train_source, y_train)
source_test_dataset  = SourceDataset(X_test_source, y_test)

source_train_loader = DataLoader(source_train_dataset, batch_size=64, shuffle=False)
source_test_loader  = DataLoader(source_test_dataset, batch_size=64, shuffle=False)

In [ ]:

class BiLSTMModel(nn.Module):
    def __init__(
        self,
        vocab_size=34000,
        embedding_dim=256,
        hidden_dim=128,
        output_dim=8
    ):
        super(BiLSTMModel, self).__init__()
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Stacked BiLSTM (2 layers)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,              # 🔥 2 layer BiLSTM
            batch_first=True,
            bidirectional=True,
            dropout=0.3               # dropout giữa các layer LSTM
        )
        
        # Regularization
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim * 2)
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_dim * 2, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        
        # Output
        self.out = nn.Linear(64, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 280)
        x = self.embedding(x)  # -> (B, 280, 286)
        
        # LSTM
        _, (h_n, _) = self.lstm(x)
        
        # Lấy hidden state cuối của 2 chiều (forward + backward)
        x = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (B, 256)
        
        return x

        # Dense
        x = self.dropout(x)
        x = self.bn1(x)
        x = self.dropout(x)
        
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.bn2(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = torch.relu(x)
        return x
        
        x = self.out(x)
        x = self.sigmoid(x)
        
        return x

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

class GAT(nn.Module):
    def __init__(
        self,
        in_channels=57,
        hidden_channels=64,
        num_classes=8
    ):
        super(GAT, self).__init__()

        # ===== GAT Layers =====
        self.conv1 = GATConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            heads=4,
            dropout=0.3
        )

        self.conv2 = GATConv(
            in_channels=hidden_channels * 4,
            out_channels=hidden_channels,
            heads=2,
            dropout=0.3
        )

        # ===== Batch Normalization =====
        self.bn1 = nn.BatchNorm1d(hidden_channels * 4)
        self.bn2 = nn.BatchNorm1d(hidden_channels * 2)

        # ===== Dense Layers =====
        self.fc1 = nn.Linear(hidden_channels * 2, hidden_channels)

        # ===== Output Layer =====
        self.fc2 = nn.Linear(hidden_channels, num_classes)

        self.dropout = 0.3

    def forward(self, data):

        # ===== Inputs =====
        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        # ===== GAT Layer 1 =====
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== GAT Layer 2 =====
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Global Pooling =====
        x = global_mean_pool(x, batch)
        return x

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Dense Layer =====
        x = self.fc1(x)
        x = F.relu(x)
        return x

        # ===== Output =====
        x = self.fc2(x)
        x = torch.sigmoid(x)

        return x

In [ ]:
codebert_model = load_model('DATASET/bert.h5')
codebert_model = Model(
    inputs=codebert_model.input,
    outputs=codebert_model.get_layer("sourcecode_features").output
)

bilstm_model = BiLSTMModel()
bilstm_model.load_state_dict(torch.load("DATASET/bilstm_model.pth"))
device = "cuda" if torch.cuda.is_available() else "cpu"
bilstm_model.to(device)
bilstm_model.eval()

gat_model = GAT()
gat_model.load_state_dict(torch.load("DATASET/gat.pth"))
gat_model.to(device)
gat_model.eval()

In [ ]:
for i, layer in enumerate(codebert_model.layers):
    try:
        print(i, layer.name, layer.output_shape)
    except:
        print(i, layer.name, "no shape")

In [ ]:
for i, layer in enumerate(codebert_model.layers):
    print(i, layer.name)
    print(layer.output.shape)

In [ ]:
for name, layer in bilstm_model.named_children():
    print(name, layer)

In [ ]:
for name, layer in gat_model.named_children():
    print(name, layer)

In [ ]:
def extract_opcode_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            out = model(x)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)  # (N, 8)


def extract_gat_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)

opcode_feat_train = extract_opcode_features(bilstm_model, opcode_train_loader, device)
opcode_feat_test  = extract_opcode_features(bilstm_model, opcode_test_loader, device)

gat_feat_train = extract_gat_features(gat_model, cfg_train_loader, device)
gat_feat_test  = extract_gat_features(gat_model, cfg_test_loader, device)

codebert_feat_train = codebert_model.predict(X_train_source, batch_size=64)
codebert_feat_test  = codebert_model.predict(X_test_source, batch_size=64)

# convert numpy -> torch nếu cần
codebert_feat_train = torch.tensor(codebert_feat_train)
codebert_feat_test  = torch.tensor(codebert_feat_test)

# train_features = torch.cat([
#     opcode_feat_train,
#     gat_feat_train,
#     codebert_feat_train
# ], dim=1)

# test_features = torch.cat([
#     opcode_feat_test,
#     gat_feat_test,
#     codebert_feat_test
# ], dim=1)

In [ ]:
# import tensorflow as tf
# from tensorflow.keras.layers import *
# from tensorflow.keras.models import Model


# # =========================================================
# # INPUTS (GIỮ ĐÚNG STRUCTURE)
# # =========================================================

# codebert_input = Input(shape=(10, 768), name="codebert_input")
# lstm_input = Input(shape=(2, 128), name="lstm_input")
# gnn_input = Input(shape=(2, 64), name="gnn_input")


# # =========================================================
# # PROJECTION (CHUẨN HÓA VỀ CÙNG DIM)
# # =========================================================

# proj_dim = 128

# codebert_tokens = Dense(proj_dim, activation="relu")(codebert_input)
# lstm_tokens = Dense(proj_dim, activation="relu")(lstm_input)
# gnn_tokens = Dense(proj_dim, activation="relu")(gnn_input)


# # =========================================================
# # TRANSFORMER BLOCK (SELF ATTENTION)
# # =========================================================

# def transformer_block(x, name):

#     attn = MultiHeadAttention(
#         num_heads=4,
#         key_dim=32,
#         dropout=0.1,
#         name=f"{name}_mha"
#     )(x, x)

#     x = Add()([x, attn])
#     x = LayerNormalization()(x)

#     ffn = Dense(256, activation="relu")(x)
#     ffn = Dropout(0.1)(ffn)
#     ffn = Dense(proj_dim)(ffn)

#     x = Add()([x, ffn])
#     x = LayerNormalization()(x)

#     return x


# # =========================================================
# # LEVEL 1: INTRA-MODAL SELF ATTENTION
# # =========================================================

# codebert_sa = transformer_block(codebert_tokens, "codebert")
# lstm_sa = transformer_block(lstm_tokens, "lstm")
# gnn_sa = transformer_block(gnn_tokens, "gnn")


# # =========================================================
# # LEVEL 2: CROSS-ATTENTION
# # =========================================================

# def cross_attention(q, kv, name):

#     attn = MultiHeadAttention(
#         num_heads=4,
#         key_dim=32,
#         dropout=0.1,
#         name=f"{name}_cross"
#     )(q, kv)

#     x = Add()([q, attn])
#     x = LayerNormalization()(x)

#     ffn = Dense(256, activation="relu")(x)
#     ffn = Dropout(0.1)(ffn)
#     ffn = Dense(proj_dim)(ffn)

#     x = Add()([x, ffn])
#     x = LayerNormalization()(x)

#     return x


# # CodeBERT ↔ LSTM
# cb_lstm = cross_attention(codebert_sa, lstm_sa, "cb_lstm")
# lstm_cb = cross_attention(lstm_sa, codebert_sa, "lstm_cb")

# # CodeBERT ↔ GNN
# cb_gnn = cross_attention(codebert_sa, gnn_sa, "cb_gnn")
# gnn_cb = cross_attention(gnn_sa, codebert_sa, "gnn_cb")

# # LSTM ↔ GNN
# lstm_gnn = cross_attention(lstm_sa, gnn_sa, "lstm_gnn")
# gnn_lstm = cross_attention(gnn_sa, lstm_sa, "gnn_lstm")


# # =========================================================
# # LEVEL 3: FUSION
# # =========================================================

# fusion = Concatenate(axis=1)([
#     cb_lstm,
#     lstm_cb,
#     cb_gnn,
#     gnn_cb,
#     lstm_gnn,
#     gnn_lstm
# ])


# fusion = Flatten()(fusion)


# # =========================================================
# # CLASSIFIER
# # =========================================================

# x = Dense(128, activation="relu")(fusion)
# x = Dropout(0.3)(x)
# x = Dense(64, activation="relu")(x)
# x = Dropout(0.3)(x)

# output = Dense(8, activation="sigmoid")(x)


# # =========================================================
# # MODEL
# # =========================================================

# fusion_model = Model(
#     inputs=[codebert_input, lstm_input, gnn_input],
#     outputs=output,
#     name="Fixed_Hierarchical_Multimodal_Transformer"
# )


# # =========================================================
# # COMPILE
# # =========================================================

# fusion_model.compile(
#     optimizer=tf.keras.optimizers.Adam(1e-4),
#     loss="binary_crossentropy",
#     metrics=[tf.keras.metrics.BinaryAccuracy(name="acc")]
# )


# fusion_model.summary()

In [ ]:
# import tensorflow as tf
# from tensorflow.keras.layers import *
# from tensorflow.keras.models import Model


# # =========================================================
# # INPUTS
# # =========================================================

# codebert_input = Input(shape=(10, 768), name="codebert_input")
# lstm_input     = Input(shape=(2, 128), name="lstm_input")
# gnn_input      = Input(shape=(2, 64), name="gnn_input")


# # =========================================================
# # PROJECT TO SAME DIM
# # =========================================================

# proj_dim = 128

# codebert = Dense(proj_dim)(codebert_input)
# lstm = Dense(proj_dim)(lstm_input)
# gnn = Dense(proj_dim)(gnn_input)


# # =========================================================
# # CROSS ATTENTION ONLY (PAIRWISE)
# # =========================================================

# def cross_attn(q, kv, name):

#     attn = MultiHeadAttention(
#         num_heads=4,
#         key_dim=32,
#         dropout=0.1,
#         name=f"{name}_cross_attn"
#     )(q, kv)

#     x = Add()([q, attn])
#     x = LayerNormalization()(x)

#     return x


# # =========================================================
# # PAIRWISE CROSS ATTENTION
# # =========================================================

# cb_lstm = cross_attn(codebert, lstm, "cb_lstm")
# cb_gnn  = cross_attn(codebert, gnn, "cb_gnn")

# lstm_cb = cross_attn(lstm, codebert, "lstm_cb")
# lstm_gnn = cross_attn(lstm, gnn, "lstm_gnn")

# gnn_cb = cross_attn(gnn, codebert, "gnn_cb")
# gnn_lstm = cross_attn(gnn, lstm, "gnn_lstm")


# # =========================================================
# # FUSION
# # =========================================================

# fusion = Concatenate(axis=1)([
#     cb_lstm,
#     lstm_cb,
#     cb_gnn,
#     gnn_cb,
#     lstm_gnn,
#     gnn_lstm
# ])


# # =========================================================
# # POOLING
# # =========================================================

# fusion = GlobalAveragePooling1D()(fusion)


# # =========================================================
# # CLASSIFIER
# # =========================================================

# x = Dense(128, activation="relu")(fusion)
# x = Dropout(0.3)(x)
# x = Dense(64, activation="relu")(x)
# x = Dropout(0.3)(x)

# output = Dense(8, activation="sigmoid")(x)


# # =========================================================
# # MODEL
# # =========================================================

# fusion_model = Model(
#     inputs=[codebert_input, lstm_input, gnn_input],
#     outputs=output
# )


# # =========================================================
# # COMPILE
# # =========================================================

# fusion_model.compile(
#     optimizer=tf.keras.optimizers.Adam(1e-4),
#     loss="binary_crossentropy",
#     metrics=[tf.keras.metrics.BinaryAccuracy()]
# )

In [27]:
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model


# =========================================================
# INPUTS (ALL HAVE 16 TOKENS)
# =========================================================

codebert_input = Input(shape=(16, 480), name="codebert_input")
lstm_input     = Input(shape=(16, 16), name="lstm_input")
gnn_input      = Input(shape=(16, 8), name="gnn_input")


# =========================================================
# PROJECT TO SAME DIMENSION
# =========================================================

proj_dim = 16

codebert = Dense(proj_dim)(codebert_input)
lstm = Dense(proj_dim)(lstm_input)
gnn = Dense(proj_dim)(gnn_input)


# =========================================================
# SELF-ATTENTION BLOCK
# =========================================================

def self_attn(x, name):
    attn = MultiHeadAttention(
        num_heads=4,
        key_dim=32,
        dropout=0.1,
        name=f"{name}_self_attn"
    )(x, x)

    x = Add()([x, attn])
    x = LayerNormalization()(x)

    return x


# =========================================================
# CROSS-ATTENTION BLOCK
# =========================================================

def cross_attn(q, kv, name):
    attn = MultiHeadAttention(
        num_heads=4,
        key_dim=32,
        dropout=0.1,
        name=f"{name}_cross_attn"
    )(q, kv)

    x = Add()([q, attn])
    x = LayerNormalization()(x)

    return x


# =========================================================
# LEVEL 1: SELF-ATTENTION
# =========================================================

codebert_sa = self_attn(codebert, "codebert")
lstm_sa = self_attn(lstm, "lstm")
gnn_sa = self_attn(gnn, "gnn")


# =========================================================
# LEVEL 2: CROSS-ATTENTION (PAIRWISE)
# =========================================================

cb_lstm = cross_attn(codebert_sa, lstm_sa, "cb_lstm")
lstm_cb = cross_attn(lstm_sa, codebert_sa, "lstm_cb")

cb_gnn = cross_attn(codebert_sa, gnn_sa, "cb_gnn")
gnn_cb = cross_attn(gnn_sa, codebert_sa, "gnn_cb")

lstm_gnn = cross_attn(lstm_sa, gnn_sa, "lstm_gnn")
gnn_lstm = cross_attn(gnn_sa, lstm_sa, "gnn_lstm")


# =========================================================
# FUSION
# =========================================================

fusion = Concatenate(axis=1)([
    cb_lstm,
    lstm_cb,
    cb_gnn,
    gnn_cb,
    lstm_gnn,
    gnn_lstm
])


# =========================================================
# GLOBAL REFINEMENT (OPTIONAL LIGHT SELF-ATTENTION)
# =========================================================

fusion = self_attn(fusion, "fusion")


# =========================================================
# POOLING
# =========================================================

fusion = GlobalAveragePooling1D()(fusion)


# =========================================================
# CLASSIFIER
# =========================================================

x = Dense(128, activation="relu")(fusion)
x = Dropout(0.3)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)

output = Dense(8, activation="sigmoid")(x)


# =========================================================
# MODEL
# =========================================================

fusion_model = Model(
    inputs=[codebert_input, lstm_input, gnn_input],
    outputs=output
)


# =========================================================
# COMPILE
# =========================================================

fusion_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy()]
)


In [28]:
codebert_feat_train = codebert_feat_train.reshape(-1, 16, 480)
codebert_feat_test  = codebert_feat_test.reshape(-1, 16, 480)

opcode_feat_train = opcode_feat_train.reshape(-1, 16, 16)
opcode_feat_test  = opcode_feat_test.reshape(-1, 16, 16)

gat_feat_train = gat_feat_train.reshape(-1, 16, 8)
gat_feat_test  = gat_feat_test.reshape(-1, 16, 8)

In [29]:
fusion_model.fit(
    [codebert_feat_train, opcode_feat_train, gat_feat_train],
    y_train,
    validation_data=([codebert_feat_test, opcode_feat_test, gat_feat_test], y_test),
    epochs= 30,
    batch_size=32
)

Epoch 1/30
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 57s 31ms/step - binary_accuracy: 0.8109 - loss: 0.4233 - val_binary_accuracy: 0.8630 - val_loss: 0.3876
Epoch 2/30
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 41s 36ms/step - binary_accuracy: 0.8672 - loss: 0.3104 - val_binary_accuracy: 0.8626 - val_loss: 0.3593
Epoch 3/30
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 38s 33ms/step - binary_accuracy: 0.8820 - loss: 0.2784 - val_binary_accuracy: 0.8675 - val_loss: 0.3414
Epoch 4/30
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 42s 37ms/step - binary_accuracy: 0.8908 - loss: 0.2581 - val_binary_accuracy: 0.8695 - val_loss: 0.3243
Epoch 5/30
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 41s 36ms/step - binary_accuracy: 0.8970 - loss: 0.2436 - val_binary_accuracy: 0.8805 - val_loss: 0.3133
Epoch 6/30
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 38s 34ms/step - binary_accuracy: 0.9026 - loss: 0.2317 - val_binary_accuracy: 0.8719 - val_loss: 0.3047
Epoch 7/30
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 51s 44ms/step - binary_accuracy: 0.9070 - loss: 0.2227 - val_binary_accuracy: 0.89

In [30]:
import numpy as np
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    roc_auc_score,
    classification_report
)

# =========================================================
# 1. EVALUATE BASIC (LOSS + ACC)
# =========================================================

loss, acc = fusion_model.evaluate(
    [codebert_feat_test, opcode_feat_test, gat_feat_test],
    y_test,
    batch_size=32,
    verbose=1
)

print("\n===== BASIC EVALUATION =====")
print("Loss:", loss)
print("Accuracy:", acc)


# =========================================================
# 2. PREDICT
# =========================================================

y_pred = fusion_model.predict(
    [codebert_feat_test, opcode_feat_test, gat_feat_test],
    batch_size=32
)

# =========================================================
# 3. BINARY THRESHOLD
# =========================================================

threshold = 0.5
y_pred_bin = (y_pred >= threshold).astype(int)


# =========================================================
# 4. MICRO / MACRO F1
# =========================================================

micro_f1 = f1_score(y_test, y_pred_bin, average="micro")
macro_f1 = f1_score(y_test, y_pred_bin, average="macro")

micro_p = precision_score(y_test, y_pred_bin, average="micro")
macro_p = precision_score(y_test, y_pred_bin, average="macro")

micro_r = recall_score(y_test, y_pred_bin, average="micro")
macro_r = recall_score(y_test, y_pred_bin, average="macro")

print("\n===== F1 / PRECISION / RECALL =====")
print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)
print("Micro Precision:", micro_p)
print("Macro Precision:", macro_p)
print("Micro Recall:", micro_r)
print("Macro Recall:", macro_r)


# =========================================================
# 5. HAMMING LOSS (VERY IMPORTANT)
# =========================================================

hl = hamming_loss(y_test, y_pred_bin)

print("\n===== HAMMING LOSS =====")
print("Hamming Loss:", hl)


# =========================================================
# 6. ROC-AUC (MULTI-LABEL)
# =========================================================

auc = roc_auc_score(y_test, y_pred, average="macro")

print("\n===== ROC-AUC =====")
print("Macro ROC-AUC:", auc)


# =========================================================
# 7. CLASSIFICATION REPORT (PER LABEL)
# =========================================================

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_test, y_pred_bin))


# =========================================================
# 8. LABEL-WISE ANALYSIS (OPTIONAL BUT VERY USEFUL)
# =========================================================

num_labels = y_test.shape[1]

print("\n===== LABEL-WISE F1 =====")
for i in range(num_labels):
    f1 = f1_score(y_test[:, i], y_pred_bin[:, i])
    print(f"Label {i}: F1 = {f1:.4f}")

285/285 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - binary_accuracy: 0.8987 - loss: 0.2468

===== BASIC EVALUATION =====
Loss: 0.24678164720535278
Accuracy: 0.8986979126930237
285/285 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step

===== F1 / PRECISION / RECALL =====
Micro F1: 0.8327563188740298
Macro F1: 0.7362095393398698
Micro Precision: 0.8643024894316581
Macro Precision: 0.7836134038418687
Micro Recall: 0.8034318648212024
Macro Recall: 0.7047944606331188

===== HAMMING LOSS =====
Hamming Loss: 0.10130208333333333

===== ROC-AUC =====
Macro ROC-AUC: 0.9321179449406541

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.86      0.64      0.74      6091
           1       0.57      0.34      0.42       476
           2       0.91      0.96      0.93      5755
           3       0.82      0.85      0.84      1268
           4       0.70      0.60      0.64      1018
           5       0.87      0.85      0.86      3798
           6       0.65     

c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
